In [ ]:
#用notebook做简单测试，可以在rag_system.py里直接装载模型
import torch
from transformers import (
    AutoModelForCausalLM,  # 用于加载因果语言模型
    AutoTokenizer,      # 用于加载分词器
    TrainingArguments,     # 训练参数配置
    BitsAndBytesConfig,    # 4位量化配置
)
from trl import SFTTrainer  # 指令微调训练器
from peft import LoraConfig, get_peft_model  # LoRA配置
from datasets import load_dataset  # 加载数据集

In [ ]:
model_path ='/public/huggingface-models/Qwen/Qwen2.5-Coder-7B'

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    use_fast=True 
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    trust_remote_code = True
)

model.generation_config.pad_token_id = tokenizer.pad_token_id

In [ ]:
message =[
{"role": "user", "content": "2001年06月03日03时，细节数据。正涡度比例是多少"}]
inputs = tokenizer.apply_chat_template(message,add_generation_prompt=True,tokenize=True,return_dict=True,return_tensors="pt",).to(model.device)
outputs = model.generate(**inputs, max_new_tokens=40)
print (tokenizer.decode(outputs [0] [inputs ["input_ids"].shape[-1]:]))
## 模型会出现幻觉：正涡度比例是0.000000000000000000000000000000000

In [ ]:
from rag_qwen_system import RAGQWENSystem

rag_system = RAGQWENSystem()
query = "" # 时间，章节（默认写“总览/摘要”）。具体问题
new_message = rag_system.get_messages_for_query(query, top_k=2)
print(new_message)

In [ ]:
# 对比二者的回答
new_message =[
{""}]
inputs = tokenizer.apply_chat_template(new_message,add_generation_prompt=True,tokenize=True,return_dict=True,return_tensors="pt",).to(model.device)
outputs = model.generate(**inputs, max_new_tokens=40)
print (tokenizer.decode(outputs [0] [inputs ["input_ids"].shape[-1]:]))